##Setup

### Import Packages and Set Constants

In [ ]:
%tensorflow_version 2.x
!pip install tensorflow -q
!pip install keras-tuner -q

import tensorflow as tf
import keras_tuner as kt
from tensorflow import keras
from tensorflow.keras import layers, models, regularizers
from sklearn.metrics import confusion_matrix, classification_report, f1_score
import seaborn as sns
from tensorflow.keras.preprocessing import image_dataset_from_directory
from google.colab import files
from IPython.display import Image, display
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import shutil
import PIL
import math
import glob
import json
import os
from tqdm import tqdm
import time
!apt-get update && apt-get -qq install xxd

In [ ]:
MODELS_DIR = 'models'
if not os.path.exists(MODELS_DIR):
  os.mkdir(MODELS_DIR)
SAVED_MODEL_FILENAME = os.path.join(MODELS_DIR, "magic_wand.keras")
FLOAT_TFL_MODEL_FILENAME = os.path.join(MODELS_DIR, "magic_wand_float.tfl")
QUANTIZED_TFL_MODEL_FILENAME = os.path.join(MODELS_DIR, "magic_wand.tfl")
TFL_CC_MODEL_FILENAME = os.path.join(MODELS_DIR, "magic_wand.cc")

DATASET_DIR =  'dataset'
if not os.path.exists(DATASET_DIR):
  os.mkdir(DATASET_DIR)
TRAIN_DIR = os.path.join(DATASET_DIR, "train")
VAL_DIR = os.path.join(DATASET_DIR, "validation")
TEST_DIR = os.path.join(DATASET_DIR, "test")
!rm -rf sample_data

CHKPT_DIR =  'checkpoints'
if not os.path.exists(CHKPT_DIR):
  os.mkdir(CHKPT_DIR)

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

TEST_PERCENTAGE = 10
VALIDATION_PERCENTAGE = 30
TRAIN_PERCENTAGE = 100 - (TEST_PERCENTAGE + VALIDATION_PERCENTAGE)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Load Your Custom Dataset
Now you'll need to upload all of your custom gesture files that you created using the Magic Wand tool (aka the ```*.json``` files). **Note: you can select multiple files and upload them all at once!**

If you are having trouble uploading files because your internet bandwidth is too slow feel free to uncomment the lines below to instead use Pete's digits dataset.

In [ ]:
# Upload your files
os.chdir("/content/dataset")
uploaded = files.upload()
os.chdir("/content")

In [ ]:
# !curl -L https://github.com/petewarden/magic_wand_digit_data/archive/8170591863f9addca27b1a963263f7c7bed33f41.zip -o magic_wand_digit_data.zip
# !unzip magic_wand_digit_data.zip
# !rm -rf magic_wand_digit_data.zip
# !mv magic_wand_digit_data-*/* dataset
# !rm -rf magic_wand_digit_data-*

**Update the variable below with the number of labeled gestures in your dataset**
Note: Use the number of unique gestures/labels and *not* the number of samples in your dataset.

In [ ]:
NUM_GESTURES = 10
# NUM_GESTURES = 2
AUGMENT_COUNT = 30

Next we'll parse the JSON files into a python object which we can more easily work with.

In [ ]:
dataset_jsons = DATASET_DIR + "/*.json"
strokes = []
for filename in sorted(glob.glob(dataset_jsons)):
  print(filename)
  with open(filename, "r") as file:
    file_contents = file.read()
  file_data = json.loads(file_contents)
  for stroke in file_data["strokes"]:
    stroke["filename"] = filename
    strokes.append(stroke)
print(len(strokes))
print(stroke["filename"])
print(dataset_jsons)

If you'd like to visualize any of your gestures you can use the helper function below!

In [ ]:
def plot_stroke(stroke):
  x_array = []
  y_array = []
  for coords in stroke["strokePoints"]:
    x_array.append(coords["x"])
    y_array.append(coords["y"])

  fig = plt.figure(figsize=(12.8, 4.8))
  fig.suptitle(stroke["label"])

  ax = fig.add_subplot(131)
  ax.set_xlabel('x')
  ax.set_ylabel('y')
  ax.set_xlim(-0.4, 0.4)
  ax.set_ylim(-0.4, 0.4)
  ax.plot(x_array, y_array)

  plt.show()

In [ ]:
# plot_stroke(strokes[980])

### Preprocess your Dataset
Next we'll preprocess the dataset to prepare it for training. By preprocessing the data in bulk before training the whole training process will execute much faster. To do so, we'll convert the strokes into rastered images using the helper functions below. This is the process used in real-time in the Arduino code to convert a gesture into an image that the CNN we are going to train can then process.

Once we have converted the dataset to rasterized images we will generate a ```Keras``` dataset for use in training.

In [ ]:
FIXED_POINT = 256

def mul_fp(a, b):
  return (a * b) / FIXED_POINT

def div_fp(a, b):
  if b == 0:
    b = 1
  return (a * FIXED_POINT) / b

def float_to_fp(a):
  return math.floor(a * FIXED_POINT)

def norm_to_coord_fp(a, range_fp, half_size_fp):
  a_fp = float_to_fp(a)
  norm_fp = div_fp(a_fp, range_fp)
  return mul_fp(norm_fp, half_size_fp) + half_size_fp

def round_fp_to_int(a):
  return math.floor((a + (FIXED_POINT / 2)) / FIXED_POINT)

def gate(a, min, max):
  if a < min:
    return min
  elif a > max:
    return max
  else:
    return a

def rasterize_stroke(stroke_points, x_range, y_range, width, height):
  num_channels = 3
  buffer_byte_count = height * width * num_channels
  buffer = bytearray(buffer_byte_count)

  width_fp = width * FIXED_POINT
  height_fp = height * FIXED_POINT
  half_width_fp = width_fp / 2
  half_height_fp = height_fp / 2
  x_range_fp = float_to_fp(x_range)
  y_range_fp = float_to_fp(y_range)

  t_inc_fp = FIXED_POINT / len(stroke_points)

  one_half_fp = (FIXED_POINT / 2)

  for point_index in range(len(stroke_points) - 1):
    start_point = stroke_points[point_index]
    end_point = stroke_points[point_index + 1]
    start_x_fp = norm_to_coord_fp(start_point["x"], x_range_fp, half_width_fp)
    start_y_fp = norm_to_coord_fp(-start_point["y"], y_range_fp, half_height_fp)
    end_x_fp = norm_to_coord_fp(end_point["x"], x_range_fp, half_width_fp)
    end_y_fp = norm_to_coord_fp(-end_point["y"], y_range_fp, half_height_fp)
    delta_x_fp = end_x_fp - start_x_fp
    delta_y_fp = end_y_fp - start_y_fp

    t_fp = point_index * t_inc_fp
    if t_fp < one_half_fp:
      local_t_fp = div_fp(t_fp, one_half_fp)
      one_minus_t_fp = FIXED_POINT - local_t_fp
      red = round_fp_to_int(one_minus_t_fp * 255)
      green = round_fp_to_int(local_t_fp * 255)
      blue = 0
    else:
      local_t_fp = div_fp(t_fp - one_half_fp, one_half_fp)
      one_minus_t_fp = FIXED_POINT - local_t_fp
      red = 0
      green = round_fp_to_int(one_minus_t_fp * 255)
      blue = round_fp_to_int(local_t_fp * 255)
    red = gate(red, 0, 255)
    green = gate(green, 0, 255)
    blue = gate(blue, 0, 255)

    if abs(delta_x_fp) > abs(delta_y_fp):
      line_length = abs(round_fp_to_int(delta_x_fp))
      if delta_x_fp > 0:
        x_inc_fp = 1 * FIXED_POINT
        y_inc_fp = div_fp(delta_y_fp, delta_x_fp)
      else:
        x_inc_fp = -1 * FIXED_POINT
        y_inc_fp = -div_fp(delta_y_fp, delta_x_fp)
    else:
      line_length = abs(round_fp_to_int(delta_y_fp))
      if delta_y_fp > 0:
        y_inc_fp = 1 * FIXED_POINT
        x_inc_fp = div_fp(delta_x_fp, delta_y_fp)
      else:
        y_inc_fp = -1 * FIXED_POINT
        x_inc_fp = -div_fp(delta_x_fp, delta_y_fp)
    for i in range(line_length + 1):
      x_fp = start_x_fp + (i * x_inc_fp)
      y_fp = start_y_fp + (i * y_inc_fp)
      x = round_fp_to_int(x_fp)
      y = round_fp_to_int(y_fp)
      if (x < 0) or (x >= width) or (y < 0) or (y >= height):
        continue
      buffer_index = (y * width * num_channels) + (x * num_channels)
      buffer[buffer_index + 0] = red
      buffer[buffer_index + 1] = green
      buffer[buffer_index + 2] = blue

  np_buffer = np.frombuffer(buffer, dtype=np.uint8).reshape(height, width, num_channels)

  return np_buffer

In [ ]:
X_RANGE = 0.6
Y_RANGE = 0.6

def ensure_empty_dir(dirname):
  dirpath = Path(dirname)
  if dirpath.exists() and dirpath.is_dir():
    shutil.rmtree(dirpath)
  dirpath.mkdir()

def augment_points(points, move_range, scale_range, rotate_range):
  move_x = np.random.uniform(low=-move_range, high=move_range)
  move_y = np.random.uniform(low=-move_range, high=move_range)
  scale = np.random.uniform(low=1.0-scale_range, high=1.0+scale_range)
  rotate = np.random.uniform(low=-rotate_range, high=rotate_range)

  x_axis_x = math.cos(rotate) * scale
  x_axis_y = math.sin(rotate) * scale

  y_axis_x = -math.sin(rotate) * scale
  y_axis_y = math.cos(rotate) * scale

  new_points = []
  for point in points:
    old_x = point["x"]
    old_y = point["y"]
    new_x = (x_axis_x * old_x) + (x_axis_y * old_y) + move_x
    new_y = (y_axis_x * old_x) + (y_axis_y * old_y) + move_y
    new_points.append({"x": new_x, "y": new_y})

  return new_points

def save_strokes_as_images(strokes, root_folder, width, height, augment_count):
  ensure_empty_dir(root_folder)
  labels = set()
  for stroke in strokes:
    labels.add(stroke["label"].lower())
  for label in labels:
    label_path = Path(root_folder, label)
    ensure_empty_dir(label_path)

  label_counts = {}
  for stroke in strokes:
    points = stroke["strokePoints"]
    label = stroke["label"].lower()
    if label == "":
      raise Exception("Missing label for %s:%d" % (stroke["filename"], stroke["index"]))
    if label not in label_counts:
      label_counts[label] = 0
    label_count = label_counts[label]
    label_counts[label] += 1
    raster = rasterize_stroke(points, X_RANGE, Y_RANGE, width, height)
    image = PIL.Image.fromarray(raster)
    image.save(Path(root_folder, label, str(label_count) + ".png"))
    for i in range(augment_count):
      augmented_points = augment_points(points, 0.1, 0.1, 0.3)
      raster = rasterize_stroke(augmented_points, X_RANGE, Y_RANGE, width, height)
      image = PIL.Image.fromarray(raster)
      image.save(Path(root_folder, label, str(label_count) + "_a" + str(i) + ".png"))
  return labels

Take the dataset and shuffle it into the Training/Validation/Test splits

In [ ]:
IMAGE_WIDTH = 32
IMAGE_HEIGHT = 32


shuffled_strokes = strokes.copy()
np.random.shuffle(shuffled_strokes)

test_count = math.floor((len(shuffled_strokes) * TEST_PERCENTAGE) / 100)
validation_count = math.floor((len(shuffled_strokes) * VALIDATION_PERCENTAGE) / 100)
test_strokes = shuffled_strokes[0:test_count]
validation_strokes = shuffled_strokes[test_count:(test_count + validation_count)]
train_strokes = shuffled_strokes[(test_count + validation_count):]

labels_test  = save_strokes_as_images(test_strokes, TEST_DIR, IMAGE_WIDTH, IMAGE_HEIGHT, AUGMENT_COUNT)
labels_val   = save_strokes_as_images(validation_strokes, VAL_DIR, IMAGE_WIDTH, IMAGE_HEIGHT, AUGMENT_COUNT)
labels_train = save_strokes_as_images(train_strokes, TRAIN_DIR, IMAGE_WIDTH, IMAGE_HEIGHT, AUGMENT_COUNT)

Also get the alphanumeric ordering of the labels as the Nueral Network will output its result in this order for the predicted class. **Make a note of this ordering as you will need to enter the labels in order in the Arduino code!**

In [ ]:
labels = sorted(labels_test.union(labels_val).union(labels_train))

labelToInt = {}
currInt = 0
for label in labels:
  labelToInt[label] = currInt
  currInt = currInt + 1
intToLabel = {v: k for k, v in labelToInt.items()}
print(intToLabel)

If you'd like to visualize the difference between a stroke and its rasterized output image, run the cell below!

In [ ]:
plot_stroke(strokes[4])
raster = rasterize_stroke(strokes[0]["strokePoints"], 0.5, 0.5, 32, 32)
PIL.Image.fromarray(raster).resize((512, 512), PIL.Image.NEAREST)

Finally, we'll generate a dataset in ```Keras```.

In [ ]:
validation_ds = image_dataset_from_directory(
    directory=VAL_DIR,
    labels='inferred',
    label_mode='categorical',
    batch_size=32,
    image_size=(IMAGE_WIDTH, IMAGE_HEIGHT)).prefetch(buffer_size=32)

train_ds = image_dataset_from_directory(
    directory=TRAIN_DIR,
    labels='inferred',
    label_mode='categorical',
    batch_size=32,
    image_size=(IMAGE_WIDTH, IMAGE_HEIGHT)).prefetch(buffer_size=32)


test_ds = image_dataset_from_directory(
    directory=TEST_DIR,
    labels='inferred',
    label_mode='categorical',
    batch_size=32,
    image_size=(IMAGE_WIDTH, IMAGE_HEIGHT)).prefetch(buffer_size=32)

In [ ]:
plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
  for i in range(9):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(images[i].numpy().astype("uint8"))
    plt.axis("off")

## Define your Model

ResNet

In [ ]:
def residual_block(x, filters, stride=1):
    shortcut = x

    x = layers.Conv2D(filters, 3, strides=stride, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    x = layers.Conv2D(filters, 3, strides=1, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)

    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, strides=stride, padding='same', use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    x = layers.Add()([x, shortcut])
    x = layers.Activation('relu')(x)

    return x

def create_resnet(input_shape=(IMAGE_WIDTH, IMAGE_HEIGHT, 3), num_classes=NUM_GESTURES):
    inputs = layers.Input(shape=input_shape)

    x = layers.Conv2D(64, 3, strides=1, padding='same', use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D(2, strides=2, padding='same')(x)

    x = residual_block(x, 64)
    x = residual_block(x, 64)
    x = residual_block(x, 128, stride=2)
    x = residual_block(x, 128)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.6)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return models.Model(inputs, outputs, name='ResNet')

model_resnet = create_resnet()
model_resnet.summary()

Yolo

In [ ]:
def create_yolo_style(input_shape=(IMAGE_WIDTH, IMAGE_HEIGHT, 3), num_classes=NUM_GESTURES):
    def conv_block(x, filters):
        half = filters // 2
        x1 = layers.Conv2D(half, 3, padding='same', use_bias=False)(x)
        x2 = layers.Conv2D(half, 5, padding='same', use_bias=False)(x)
        x  = layers.Concatenate()([x1, x2])
        x  = layers.BatchNormalization()(x)
        x  = layers.LeakyReLU(negative_slope=0.1)(x)
        return x

    def channel_attention(x, ratio=4):
        c  = int(x.shape[-1])
        se = layers.GlobalAveragePooling2D()(x)
        se = layers.Reshape((1, 1, c))(se)
        se = layers.Conv2D(max(1, c // ratio), 1, activation='swish')(se)
        se = layers.Conv2D(c, 1, activation='sigmoid')(se)
        return layers.Multiply()([x, se])

    inputs = layers.Input(shape=input_shape)

    x = conv_block(inputs, 32)
    x = layers.MaxPooling2D(2, 2)(x)

    x = conv_block(x, 64)
    x = layers.MaxPooling2D(2, 2)(x)

    x = conv_block(x, 128)
    x = layers.MaxPooling2D(2, 2)(x)

    x = conv_block(x, 256)
    x = layers.MaxPooling2D(2, 2)(x)

    x = conv_block(x, 512)
    x = channel_attention(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(512)(x)
    x = layers.LeakyReLU(negative_slope=0.1)(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return models.Model(inputs, outputs, name='YOLO_Style')

model_yolo = create_yolo_style()
model_yolo.summary()

MobileViT

In [ ]:

def mobile_vit_block(x, expansion=4, transformer_dim=16, num_transformer_blocks=2, num_heads=4):
    shortcut = x
    in_channels = int(x.shape[-1])

    x = layers.Conv2D(expansion * in_channels, 1, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('swish')(x)

    x = layers.DepthwiseConv2D(3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('swish')(x)

    x = layers.Conv2D(transformer_dim, 1, use_bias=False)(x)
    x = layers.BatchNormalization()(x)

    h = int(x.shape[1])
    w = int(x.shape[2])
    c = int(x.shape[3])

    x = layers.Reshape((h * w, c))(x)

    for _ in range(num_transformer_blocks):
        x1 = layers.LayerNormalization(epsilon=1e-6)(x)
        attn = layers.MultiHeadAttention(
            num_heads=min(num_heads, max(1, c // 4)),
            key_dim=max(1, c // max(1, min(num_heads, max(1, c // 4))))
        )(x1, x1)
        x = layers.Add()([x, attn])

        x2 = layers.LayerNormalization(epsilon=1e-6)(x)
        mlp = layers.Dense(c * 2, activation='swish')(x2)
        mlp = layers.Dense(c)(mlp)
        x = layers.Add()([x, mlp])

    x = layers.Reshape((h, w, c))(x)

    x = layers.Conv2D(in_channels, 1, use_bias=False)(x)
    x = layers.BatchNormalization()(x)

    x = layers.Add()([x, shortcut])
    return x

def create_mobilevit(input_shape=(IMAGE_WIDTH, IMAGE_HEIGHT, 3), num_classes=NUM_GESTURES):
    inputs = layers.Input(shape=input_shape)

    x = layers.Conv2D(16, 3, strides=2, padding='same', use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('swish')(x)

    x = mobile_vit_block(x, expansion=2, transformer_dim=16, num_transformer_blocks=1)
    x = layers.MaxPooling2D(2, 2)(x)

    x = mobile_vit_block(x, expansion=4, transformer_dim=24, num_transformer_blocks=2)
    x = layers.MaxPooling2D(2, 2)(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='swish')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return models.Model(inputs, outputs, name='MobileViT')

model_mobilevit = create_mobilevit()
model_mobilevit.summary()


EfficientNetV2

In [ ]:
def mbconv_block(x, filters, expansion, stride):
    shortcut = x
    in_channels = int(x.shape[-1])

    x = layers.Conv2D(expansion * in_channels, 1, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('swish')(x)

    x = layers.DepthwiseConv2D(3, strides=stride, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('swish')(x)

    se = layers.GlobalAveragePooling2D()(x)
    se = layers.Reshape((1, 1, expansion * in_channels))(se)
    se = layers.Conv2D(max(1, expansion * in_channels // 4), 1, activation='swish')(se)
    se = layers.Conv2D(expansion * in_channels, 1, activation='sigmoid')(se)
    x = layers.Multiply()([x, se])

    x = layers.Conv2D(filters, 1, use_bias=False)(x)
    x = layers.BatchNormalization()(x)

    if stride == 1 and in_channels == filters:
        x = layers.Add()([x, shortcut])

    return x

def create_efficientnetv2(input_shape=(IMAGE_WIDTH, IMAGE_HEIGHT, 3), num_classes=NUM_GESTURES):
    inputs = layers.Input(shape=input_shape)

    x = layers.Conv2D(32, 3, strides=2, padding='same', use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('swish')(x)

    x = mbconv_block(x, 16, 1, 1)
    x = mbconv_block(x, 24, 6, 2)
    x = mbconv_block(x, 24, 6, 1)
    x = mbconv_block(x, 48, 6, 2)
    x = mbconv_block(x, 48, 6, 1)
    x = mbconv_block(x, 64, 6, 2)
    x = mbconv_block(x, 64, 6, 1)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(320, activation='swish')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return models.Model(inputs, outputs, name='EfficientNetV2')

model_efficientnet = create_efficientnetv2()
model_efficientnet.summary()

ConvNeXt

In [ ]:

def convnext_block(x, filters):
    shortcut = x
    in_channels = int(x.shape[-1])

    x = layers.DepthwiseConv2D(7, padding='same', use_bias=False)(x)
    x = layers.LayerNormalization(epsilon=1e-6)(x)

    x = layers.Conv2D(filters * 4, 1, use_bias=False)(x)
    x = layers.Activation('gelu')(x)

    def grn_fn(t):
        gx = keras.ops.mean(keras.ops.square(t), axis=(1, 2), keepdims=True)
        nx = keras.ops.sqrt(gx + 1e-6)
        return t / nx

    x = layers.Lambda(grn_fn)(x)

    x = layers.Conv2D(filters, 1, use_bias=False)(x)

    if in_channels != filters:
        shortcut = layers.Conv2D(filters, 1, padding='same', use_bias=False)(shortcut)

    x = layers.Add()([x, shortcut])
    return x

def create_convnext(input_shape=(IMAGE_WIDTH, IMAGE_HEIGHT, 3), num_classes=NUM_GESTURES):
    inputs = layers.Input(shape=input_shape)

    x = layers.Conv2D(64, 4, strides=4, padding='same')(inputs)
    x = layers.LayerNormalization(epsilon=1e-6)(x)

    for _ in range(3):
        x = convnext_block(x, 64)
    x = layers.MaxPooling2D(2, 2, padding='same')(x)

    x = layers.Conv2D(128, 1, padding='same')(x)
    for _ in range(3):
        x = convnext_block(x, 128)
    x = layers.MaxPooling2D(2, 2, padding='same')(x)

    x = layers.Conv2D(256, 1, padding='same')(x)
    for _ in range(3):
        x = convnext_block(x, 256)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    x = layers.Dense(256, activation='gelu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return models.Model(inputs, outputs, name='ConvNeXt')

model_convnext = create_convnext()
model_convnext.summary()


## Train your Model

Now that we have a preprocessed dataset and a model its time to train that model with that dataset!

In [ ]:
EPOCHS = 40

checkpointFileLoc = CHKPT_DIR + "/save_at_{epoch:02d}.h5"

def train_model(model, model_name, validation_ds, train_ds, epochs=EPOCHS, batch_size=64, lr=0.001):
    print(f'\nTraining model{model_name}')

    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy', keras.metrics.Precision(name='precision'), keras.metrics.Recall(name='recall')]
    )

    callbacks = [
        keras.callbacks.ReduceLROnPlateau('val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
        keras.callbacks.EarlyStopping('val_accuracy', patience=5, restore_best_weights=True, verbose=1),
        keras.callbacks.ModelCheckpoint(checkpointFileLoc, 'val_accuracy', save_best_only=True, verbose=0)
    ]

    start_time = time.time()
    history = model.fit(
        train_ds,
        validation_data=validation_ds,
        epochs=epochs,
        callbacks=callbacks,
        verbose=1
    )
    train_time = time.time() - start_time

    test_results = model.evaluate(validation_ds, verbose=0)
    val_acc = test_results[1]
    num_samples = sum(x.shape[0] for x, _ in train_ds)
    samples_per_sec = num_samples / train_time if train_time > 0 else 0


    print(f'\n{model_name} Done')
    print(f' Training time: {train_time:.2f}s')
    print(f' Val acc: {val_acc*100:.2f}%')
    print(f' Params num: {model.count_params():,}')
    print(f' Throughput: {samples_per_sec:.1f} samples/s')

    return history, val_acc, train_time, samples_per_sec

In [ ]:
models_config = [
    (model_resnet, 'ResNet'),
    (model_yolo, 'YOLO_Style'),
    (model_mobilevit, 'MobileViT'),
    (model_efficientnet, 'EfficientNetV2'),
    (model_convnext, 'ConvNeXt')
]

all_results = []
trained_models = {}

for model, name in models_config:
  history, val_acc, train_time, sps = train_model(
    model, name, validation_ds ,train_ds,
    epochs=EPOCHS
  )

  all_results.append({
      'name': name,
      'params': model.count_params(),
      'val_acc': val_acc,
      'train_time': train_time,
      'samples_per_sec': sps,
      'history': history
  })
  trained_models[name] = model


Accuracy vs Model Size Trade-off
and Final Validation Accuracy by Architecture

In [ ]:
plt.figure(figsize=(18, 5))

plt.subplot(1, 3, 1)
for r in all_results:
    plt.plot(r['history'].history['accuracy'], label=r['name'])
plt.xlabel('Epoch')
plt.ylabel('Training Accuracy')
plt.title('Training Accuracy per Epoch')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 2)
for r in all_results:
    plt.plot(r['history'].history['val_accuracy'], label=r['name'])
plt.xlabel('Epoch')
plt.ylabel('Validation Accuracy')
plt.title('Validation Accuracy per Epoch')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 3)
names = [r['name'] for r in all_results]
accs = [r['val_acc'] * 100 for r in all_results]
colors = plt.cm.viridis(np.linspace(0, 1, len(names)))
bars = plt.bar(names, accs, color=colors)
plt.ylabel('Validation Accuracy (%)')
plt.title('Final Validation Accuracy by Architecture')
plt.xticks(rotation=45)
for bar, acc in zip(bars, accs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{acc:.1f}%', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('architecture_comparison.png', dpi=150, bbox_inches='tight')

plt.figure(figsize=(8, 5))
for r in all_results:
    plt.scatter(r['params']/1e3, r['val_acc']*100, s=100)
    plt.annotate(r['name'], (r['params']/1e3, r['val_acc']*100),
                 textcoords="offset points", xytext=(5, 5))
plt.xlabel('Parameters (K)')
plt.ylabel('Validation Accuracy (%)')
plt.title('Accuracy vs Model Size Trade-off')
plt.grid(True, alpha=0.3)
plt.savefig('accuracy_vs_size.png', dpi=150)
plt.show()

In [ ]:
header = f'{"Architecture":<16} {"Params":>12} {"Val Acc":>10} {"Train Time":>12} {"Samples/s":>12}'
print('Architecture Comparison Summary')
print('=' * 64)
print(header)
print('-' * 64)

for r in all_results:
  print(f'{r["name"]:<16} {r["params"]:>12,} {r["val_acc"]*100:>9.2f}%'f' {r["train_time"]:>11.2f}s {r["samples_per_sec"]:>12.1f}')

print('='*64)

best_acc = max(all_results, key=lambda x: x['val_acc'])
fastest = min(all_results, key=lambda x: x['train_time'])
smallest = min(all_results, key=lambda x: x['params'])

print(f'\nHighest acc {best_acc["name"]} ({best_acc["val_acc"]*100:.2f}%)')
print(f'Fastest Train：{fastest["name"]} ({fastest["train_time"]:.2f}s)')
print(f'Fewest params：{smallest["name"]} ({smallest["params"]:,} params)')

## Model Selection Justification

The model selected for TFLite conversion and Arduino deployment is
**EfficientNetV2**, based on the following analysis of the comparison table.

### Accuracy is effectively equal across all models

All five architectures achieved validation accuracy between 98.45% and 98.74%,
a difference of less than 0.3 percentage points. At this scale, the gap is
within normal variance and cannot justify selecting a heavier model purely
on accuracy grounds. The deployment constraints therefore become the
deciding factor.

### Flash memory eliminates three candidates

The Arduino Nano 33 BLE Sense provides 1 MB of Flash for model storage.
Under INT8 quantization, model size in bytes is approximately equal to
parameter count.
YOLO_Style and ConvNeXt are immediately eliminated — their quantized sizes
exceed the Flash limit by 3× and 2× respectively, making on-device
deployment impossible regardless of accuracy.

MobileViT fits in Flash but contains MultiHeadAttention operations that
are not fully supported by the TFLite INT8 runtime. Conversion either
fails or falls back to a slow reference kernel, making reliable real-time
inference on the microcontroller impossible.

ResNet fits within Flash but at 713 KB leaves very little headroom. Any
additional model metadata or quantization overhead risks exceeding the
limit on the actual device.

### EfficientNetV2 is best model

EfficientNetV2 achieves 98.62% validation accuracy while using only
365,506 parameters — less than half of ResNet's parameter count —
leaving a comfortable 634 KB of Flash headroom. Its architecture
consists entirely of standard Conv2D, DepthwiseConv2D, BatchNormalization,
and Squeeze-Excitation operations, all of which the TFLite INT8 kernel
supports natively without fallback. It also trained the fastest of all
five models at 1541 seconds with 12.9 samples per second, confirming
its computational efficiency.


In [ ]:
best_model_name = "EfficientNetV2"
best_model = trained_models[best_model_name]
test_results = best_model.evaluate(test_ds)
test_loss = test_results[0]
test_acc = test_results[1]
print(f"best_model_name: {best_model_name}")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")


Confusion Matrix

In [ ]:
y_true, y_pred = [], []
for images, labels_batch in test_ds:
    preds = best_model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels_batch.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=list(intToLabel.values()),
            yticklabels=list(intToLabel.values()))
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix — {best_model_name}')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()


print(classification_report(y_true, y_pred,
                             target_names=list(intToLabel.values())))
f1 = f1_score(y_true, y_pred, average='weighted')
print(f'Weighted F1 Score: {f1:.4f}')

In [ ]:
def build_tunable_efficientnetv2(hp):
  stem_filters = hp.Choice('stem_filters',[16, 24, 32])
  expansion_rate = hp.Choice('expansion_rate',[4, 6])
  se_ratio = hp.Choice('se_ratio',[4, 8, 16])
  dropout_rate = hp.Float ('dropout',min_value=0.2,max_value=0.6, step=0.1)
  dense_units = hp.Choice('dense_units',[128, 256, 320])
  learning_rate = hp.Choice('lr',[1e-2, 1e-3, 5e-4, 1e-4])
  l2_reg = hp.Choice('l2_reg',[0.0, 1e-4, 1e-3])

  reg = regularizers.l2(l2_reg) if l2_reg > 0 else None

  def mbconv_block(x, filters, expansion, stride):
      shortcut = x
      in_channels = int(x.shape[-1])

      x = layers.Conv2D(expansion * in_channels, 1,
                        use_bias=False)(x)
      x = layers.BatchNormalization()(x)
      x = layers.Activation('swish')(x)

      x = layers.DepthwiseConv2D(3, strides=stride,
                                  padding='same', use_bias=False)(x)
      x = layers.BatchNormalization()(x)
      x = layers.Activation('swish')(x)

      se = layers.GlobalAveragePooling2D()(x)
      se = layers.Reshape((1, 1, expansion * in_channels))(se)
      se = layers.Conv2D(
              max(1, expansion * in_channels // se_ratio),
              1, activation='swish')(se)
      se = layers.Conv2D(expansion * in_channels, 1,
                          activation='sigmoid')(se)
      x  = layers.Multiply()([x, se])

      x = layers.Conv2D(filters, 1, use_bias=False)(x)
      x = layers.BatchNormalization()(x)

      if stride == 1 and in_channels == filters:
          x = layers.Add()([x, shortcut])

      return x

  inputs = layers.Input(shape=(IMAGE_WIDTH, IMAGE_HEIGHT, 3))

  x = layers.Conv2D(stem_filters, 3, strides=2, padding='same', use_bias=False)(inputs)
  x = layers.BatchNormalization()(x)
  x = layers.Activation('swish')(x)

  x = mbconv_block(x, 16, 1, 1)
  x = mbconv_block(x, 24, expansion_rate, 2)
  x = mbconv_block(x, 24, expansion_rate, 1)
  x = mbconv_block(x, 48, expansion_rate, 2)
  x = mbconv_block(x, 48, expansion_rate, 1)
  x = mbconv_block(x, 64, expansion_rate, 2)
  x = mbconv_block(x, 64, expansion_rate, 1)

  x = layers.GlobalAveragePooling2D()(x)
  x = layers.Dense(dense_units, activation='swish', kernel_regularizer=reg)(x)
  x = layers.Dropout(dropout_rate)(x)
  outputs = layers.Dense(NUM_GESTURES, activation='softmax')(x)

  model = models.Model(inputs, outputs, name='EfficientNetV2_tuned')
  model.compile(
      optimizer=keras.optimizers.Adam(learning_rate),
      loss='categorical_crossentropy',
      metrics=['accuracy', keras.metrics.Precision(name='precision'), keras.metrics.Recall(name='recall')]
  )
  return model


tuner = kt.BayesianOptimization(
    build_tunable_efficientnetv2,
    objective=kt.Objective('val_accuracy', direction='max'),
    max_trials=20,
    num_initial_points=5,
    seed=SEED,
    directory='tuner_results',
    project_name='efficientnetv2_gesture',
    overwrite=True
)

tuner.search_space_summary()

print("\nStarting Bayesian search...")
tuner.search(
    train_ds,
    validation_data=validation_ds,
    epochs=15,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=3,
            restore_best_weights=True
        )
    ],
    verbose=2
)

In [ ]:
tuner.results_summary(num_trials=5)
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print("\nBest hyperparameters found:")
print(f"  stem_filters   : {best_hps.get('stem_filters')}")
print(f"  expansion_rate : {best_hps.get('expansion_rate')}")
print(f"  se_ratio       : {best_hps.get('se_ratio')}")
print(f"  dropout        : {best_hps.get('dropout'):.1f}")
print(f"  dense_units    : {best_hps.get('dense_units')}")
print(f"  learning_rate  : {best_hps.get('lr')}")
print(f"  l2_reg         : {best_hps.get('l2_reg')}")


print("\nRetraining with best hyperparameters...")
best_model_tuned = tuner.hypermodel.build(best_hps)

history_tuned, val_acc_tuned, train_time_tuned, sps_tuned = train_model(
    best_model_tuned,
    'EfficientNetV2_tuned',
    validation_ds,
    train_ds,
    epochs=EPOCHS
)


baseline = next(r for r in all_results if r['name'] == 'EfficientNetV2')

print("\n" + "=" * 52)
print("Bayesian Tuning Impact")
print("=" * 52)
print(f"{'':22} {'Baseline':>12} {'Tuned':>12}")
print("-" * 52)
print(f"{'Val accuracy':<22} "
      f"{baseline['val_acc']*100:>11.2f}% "
      f"{val_acc_tuned*100:>11.2f}%")
print(f"{'Parameters':<22} "
      f"{baseline['params']:>12,} "
      f"{best_model_tuned.count_params():>12,}")
print(f"{'Est. Flash (INT8)':<22} "
      f"{baseline['params']/1024:>10.0f}KB "
      f"{best_model_tuned.count_params()/1024:>10.0f}KB")
print(f"{'Train time':<22} "
      f"{baseline['train_time']:>11.2f}s "
      f"{train_time_tuned:>11.2f}s")
print("=" * 52)

best_model = best_model_tuned
best_model_name = 'EfficientNetV2_tuned'

trials_data = []
for trial in tuner.oracle.trials.values():
    if trial.score is not None:
        hp = trial.hyperparameters
        trials_data.append({
            'trial_id' : int(trial.trial_id),
            'val_acc' : trial.score,
            'lr' : hp.get('lr'),
            'dropout' : hp.get('dropout'),
            'dense' : hp.get('dense_units'),
            'expansion' : hp.get('expansion_rate'),
        })

df_trials = pd.DataFrame(trials_data).sort_values('trial_id')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(df_trials['trial_id'], df_trials['val_acc'] * 100, 'o-', color='#1D9E75', linewidth=1.5)
axes[0].axhline(y=df_trials['val_acc'].max() * 100, color='#E24B4A', linestyle='--', linewidth=1, label=f"Best: {df_trials['val_acc'].max()*100:.2f}%")
axes[0].set_xlabel('Trial')
axes[0].set_ylabel('Validation accuracy (%)')
axes[0].set_title('Bayesian search convergence')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

sc = axes[1].scatter(df_trials['lr'], df_trials['dropout'], c=df_trials['val_acc'] * 100, cmap='YlGn', s=80, edgecolors='gray', linewidths=0.5)
plt.colorbar(sc, ax=axes[1], label='Val accuracy (%)')
axes[1].set_xscale('log')
axes[1].set_xlabel('Learning rate')
axes[1].set_ylabel('Dropout rate')
axes[1].set_title('LR vs dropout exploration')
axes[1].grid(True, alpha=0.3)

labels = ['Baseline\nEfficientNetV2', 'Tuned\nEfficientNetV2']
accs = [baseline['val_acc'] * 100, val_acc_tuned * 100]
colors = ['#B4B2A9', '#1D9E75']
bars = axes[2].bar(labels, accs, color=colors, width=0.4)
axes[2].set_ylabel('Validation accuracy (%)')
axes[2].set_title('Before vs after tuning')
axes[2].set_ylim(min(accs) - 1, 101)
for bar, acc in zip(bars, accs):
  axes[2].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1, f'{acc:.2f}%', ha='center', va='bottom', fontsize=10)
axes[2].grid(True, alpha=0.3, axis='y')

plt.suptitle('EfficientNetV2 — Bayesian hyperparameter search', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('bayesian_search_efficientnetv2.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
best_model.save(SAVED_MODEL_FILENAME)
print(f'Saved best model ({best_model_name}) to {SAVED_MODEL_FILENAME}')

## Test your TensorFlow Model

Lets now test out the TF model on the test dataset. We'll print out any gesture we get wrong as well as the percentage of known gestures correct as well as the number of gestures that were marked as unknown.

In [ ]:
SCORE_THRESHOLD = 0.75

def predict_image(model, filename):
  img = keras.preprocessing.image.load_img(filename, target_size=(IMAGE_WIDTH, IMAGE_HEIGHT))
  img_array = keras.preprocessing.image.img_to_array(img)
  img_array = tf.expand_dims(img_array, 0)
  predictions = model.predict(img_array).flatten()
  predicted_label_index = np.argmax(predictions)
  predicted_score = predictions[predicted_label_index]
  return (predicted_label_index, predicted_score)

correct_count = 0
wrong_count = 0
discarded_count = 0
for label_dir in glob.glob(TEST_DIR + "/*"):
  label = label_dir.replace(TEST_DIR + "/", "")
  print("Testing Gesture: ",label," with datasize: ",len(glob.glob(label_dir + "/*.png")))
  for filename in glob.glob(label_dir + "/*.png"):
    # TODO
    index, score = predict_image(best_model, filename)
    if score < SCORE_THRESHOLD:
      discarded_count += 1
      continue
    if index == labelToInt[label]:
      correct_count += 1
    else:
      wrong_count += 1
      print(label,index,score)
      print("[%s] expected, [%s] found with score [%f]" % (label, intToLabel[index], score))
      display(Image(filename=filename))

if correct_count + wrong_count == 0:
  print("All images marked as unknown!")
else:
  correct_percentage = (correct_count / (correct_count + wrong_count)) * 100
  print("%.1f%% correct (N=%d, %d unknown)" % (correct_percentage, (correct_count + wrong_count), discarded_count))

## Generate a TensorFlow Lite Model

Convert the frozen graph into a TensorFlow Lite model, which is fully quantized for use with embedded devices. The following cell will also print the model size.

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
model_no_quant_tflite = converter.convert()
open(FLOAT_TFL_MODEL_FILENAME, "wb").write(model_no_quant_tflite)


# def representative_dataset():
#   for filename in glob.glob(TEST_DIR + "/*/*.png"):
#     img = keras.preprocessing.image.load_img(filename, target_size=(IMAGE_WIDTH, IMAGE_HEIGHT))
#     img_array = keras.preprocessing.image.img_to_array(img)
#     img_array = tf.expand_dims(img_array, 0)  # Create batch axis for images, labels in train_ds.take(1):
#     yield([img_array])

def representative_dataset():
  for img_batch, _ in train_ds.take(100):
    for i in range(img_batch.shape[0]):
      sample = tf.expand_dims(img_batch[i], 0)
      yield [sample]
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
converter.representative_dataset = representative_dataset
model_tflite = converter.convert()

open(QUANTIZED_TFL_MODEL_FILENAME, "wb").write(model_tflite)

Compare the sizes of the Tensorflow, TensorFlow Lite and Quantized TensorFlow Lite models.

In [ ]:
def get_dir_size(dir):
  size = 0
  for f in os.scandir(dir):
    if f.is_file():
      size += f.stat().st_size
    elif f.is_dir():
      size += get_dir_size(f.path)
  return size

size_tf = os.path.getsize(SAVED_MODEL_FILENAME)
size_no_quant_tflite = os.path.getsize(FLOAT_TFL_MODEL_FILENAME)
size_tflite = os.path.getsize(QUANTIZED_TFL_MODEL_FILENAME)

pd.DataFrame.from_records(
    [["TensorFlow", f"{size_tf} bytes", ""],
     ["TensorFlow Lite", f"{size_no_quant_tflite} bytes ", f"(reduced by {size_tf - size_no_quant_tflite} bytes)"],
     ["TensorFlow Lite Quantized", f"{size_tflite} bytes", f"(reduced by {size_no_quant_tflite - size_tflite} bytes)"]],
     columns = ["Model", "Size", ""], index="Model")

## Test your TensorFlow Lite Models

Lets now test out the TFLite models (quantized and unquantized) on the test dataset. We'll print out any gesture we get wrong as well as the percentage of known gestures correct as well as the number of gestures that were marked as unknown.

In [ ]:
def predict_tflite(tflite_model, filename):
  img = keras.preprocessing.image.load_img(filename, target_size=(IMAGE_WIDTH, IMAGE_HEIGHT))
  img_array = keras.preprocessing.image.img_to_array(img)
  img_array = tf.expand_dims(img_array, 0)

  interpreter = tf.lite.Interpreter(model_content=tflite_model)
  interpreter.allocate_tensors()

  input_details = interpreter.get_input_details()[0]
  output_details = interpreter.get_output_details()[0]

  input_scale, input_zero_point = input_details["quantization"]
  if (input_scale, input_zero_point) != (0.0, 0):
    img_array = np.multiply(img_array, 1.0 / input_scale) + input_zero_point
    img_array = img_array.astype(input_details["dtype"])

  interpreter.set_tensor(input_details["index"], img_array)
  interpreter.invoke()
  pred = interpreter.get_tensor(output_details["index"])[0]

  output_scale, output_zero_point = output_details["quantization"]
  if (output_scale, output_zero_point) != (0.0, 0):
    pred = pred.astype(np.float32)
    pred = np.multiply((pred - output_zero_point), output_scale)

  predicted_label_index = np.argmax(pred)
  predicted_score = pred[predicted_label_index]
  return (predicted_label_index, predicted_score)

In [ ]:
def run_tflite_test(model_file):
  correct_count = 0
  wrong_count = 0
  discarded_count = 0
  for label_dir in glob.glob(TEST_DIR + "/*"):
    label = label_dir.replace(TEST_DIR + "/", "")
    print("Testing Gesture: ",label," with datasize: ",len(glob.glob(label_dir + "/*.png")))
    for filename in glob.glob(label_dir + "/*.png"):
      index, score = predict_tflite(model_file, filename)
      if score < SCORE_THRESHOLD:
        discarded_count += 1
        continue
      if index == labelToInt[label]:
        correct_count += 1
      else:
        wrong_count += 1
        print("[%s] expected, [%s] found with score [%f]" % (label, intToLabel[index], score))
        display(Image(filename=filename))

  total = correct_count + wrong_count
  if total == 0:
    print('All images discarded as unknown (score < %.2f).' % SCORE_THRESHOLD)
  else:
    print('%.1f%% correct (N=%d, %d unknown/discarded)' %
              (correct_count / total * 100, total, discarded_count))

First test the float model.

In [ ]:
run_tflite_test(model_no_quant_tflite)

Then test the quantized model

In [ ]:
run_tflite_test(model_tflite)

In [ ]:
# TEST_IMAGE = # UPDATE ME e.g., "test/0/1.png"
# index, score = predict_tflite(model_no_quant_tflite, TEST_IMAGE)
# print("Float model result:")
# print(index, score) # prints the guessed index and the confidence
# index, score = predict_tflite(model_tflite, TEST_IMAGE)
# print("Quantized model result:")
# print(index, score) # prints the guessed index and the confidence

## Generate a TensorFlow Lite for Microcontrollers Model
To convert the TensorFlow Lite quantized model into a C source file that can be loaded by TensorFlow Lite for Microcontrollers on Arduino we simply need to use the ```xxd``` tool to convert the ```.tflite``` file into a ```.cc``` file.

In [ ]:
# # Convert to a C source file, i.e, a TensorFlow Lite for Microcontrollers model
# !xxd -i {QUANTIZED_TFL_MODEL_FILENAME} > {TFL_CC_MODEL_FILENAME}
# # Update variable names
# REPLACE_TEXT = QUANTIZED_TFL_MODEL_FILENAME.replace('/', '_').replace('.', '_')
# !sed -i 's/'{REPLACE_TEXT}'/g_magic_wand_model_data/g' {TFL_CC_MODEL_FILENAME}

import subprocess

replace_text = QUANTIZED_TFL_MODEL_FILENAME.replace('/', '_').replace('.', '_')

result = subprocess.run(
  ['xxd', '-i', QUANTIZED_TFL_MODEL_FILENAME],
  capture_output=True, text=True
)
if result.returncode != 0:
  raise RuntimeError(f'xxd failed: {result.stderr}')

cc_content = result.stdout.replace(replace_text, 'g_magic_wand_model_data')

with open(TFL_CC_MODEL_FILENAME, 'w') as f:
  f.write(cc_content)

print(f'C array written to {TFL_CC_MODEL_FILENAME} ({len(cc_content):,} chars)')


That's it! You've successfully converted your TensorFlow Lite model into a TensorFlow Lite for Microcontrollers model! Run the cell below to print out its contents which we'll need for our next step, deploying the model using the Arudino IDE!

In [ ]:
# # Print the C source file
# !cat {TFL_CC_MODEL_FILENAME}
# # !tail {TFL_CC_MODEL_FILENAME} # run this command to just see the end of the file (aka the size)

with open(TFL_CC_MODEL_FILENAME, 'r') as f:
    cc_text = f.read()

preview_lines = cc_text.splitlines()[:40]
print('\n'.join(preview_lines))
print(f'... ({len(cc_text.splitlines())} lines total)')


To download your model for use at a later date:

1. On the left of the UI click on the folder icon
2. Click on the three dots to the right of the ```.cc``` file you just generated and select "download." The file can be found at ```models/{TFL_CC_MODEL_FILENAME}``` which by default is ```models/magic_wand.cc```

Next we'll deploy that model using the Arduino IDE.